# **PART A – REGRESSION**

**Medical Cost Prediction**

**Objective**: Build a Linear Regression model to predict medical insurance charges.

**SECTION 1 –** Data Understanding (Mandatory).

1. Load the dataset and display:

○ Shape

○ Column names

○ Data types

○ First 5 rows

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load dataset (adjust filename/path as needed)
df = pd.read_csv("/content/sample_data/insurance.csv")  # typical dataset name


In [ ]:
# Shape
print("Shape:", df.shape)

In [ ]:
# Column names
print("\nColumns:", df.columns.tolist())

In [ ]:
# Data types
print("\nData types:")
print(df.dtypes)

In [ ]:
# First 5 rows
print("\nFirst 5 rows:")
print(df.head())

Typical columns: age, sex, bmi, children, smoker, region, charges.

Typical data‑type mix: integer (age, children), float (bmi, charges), object/categorical (sex, smoker, region).

**2. Check:**

Missing values and duplicates

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

In most variants of this dataset, there are no missing values and a small number of duplicates (often 1–2) that you drop.

If duplicates exist, remove them with:

In [ ]:
df.drop_duplicates(inplace=True)


**3. Perform EDA:**

○ Distribution of charges

○ Scatter plot: age vs charges

○ Boxplot: smoker vs charges

○ Correlation heatmap

* Distribution of charges


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["charges"], kde=True)
plt.title("Distribution of Medical Charges")
plt.xlabel("Charges")
plt.show()


Charges are usually right‑skewed (most people have low charges; a few have very high values)



*   Scatter plot: age vs charges




In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="charges", alpha=0.7)
plt.title("Age vs Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()




 You typically see that charges increase with age, especially for certain subgroups (e.g., smokers).   




* Boxplot: smoker vs charges

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Smoker vs Charges")
plt.xlabel("Smoker (yes/no)")
plt.ylabel("Charges")
plt.show()


Smokers usually have much higher median and upper‑tail charges than non‑smokers.

* Correlation heatmap

In [ ]:
# Convert categorical columns to numeric (e.g., 1/0) for numeric correlation
df_numeric = df.copy()
df_numeric["sex"] = df_numeric["sex"].map({"female": 0, "male": 1})
df_numeric["smoker"] = df_numeric["smoker"].map({"no": 0, "yes": 1})
df_numeric["region"] = df_numeric["region"].astype("category").cat.codes

In [ ]:
# Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

The strongest correlation is usually smoker vs charges (positive), while other numeric features (age, bmi, children) show weaker positive links.

**4. Three observations from EDA**

* Charges are right‑skewed, meaning most policyholders pay relatively low amounts, but a small group contributes very high charges; this suggests the need for log‑transformation or robust regression handling.

* Smoking status strongly influences charges: smokers have significantly higher median and maximum charges than non‑smokers, indicating that smoker will be a key predictor in the regression model.

* Age and charges are positively related: older individuals tend to have higher medical costs, and the relationship becomes more pronounced when split by smoking status (older smokers show the highest charges).



**SECTION 2 – Data Preprocessing.**

* **Encode categorical variables properly**

Common categorical columns in this dataset are:
sex, smoker, region.

Use one‑hot encoding (with drop_first=True to avoid multicollinearity):

In [ ]:
import pandas as pd

# Assuming df is your cleaned DataFrame from Section 1
df_processed = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True
)


sex_male (1 = male, 0 = female), smoker_yes (1 = smoker, 0 = non‑smoker), and region dummy columns are created.

This keeps numerical features (age, bmi, children) intact and converts all categorical text into numeric flags.

*  Separate features (X) and target (y).

In [ ]:
X = df_processed.drop("charges", axis=1)   # all feature columns
y = df_processed["charges"]                # target (charges)


X now contains age, bmi, children, and all one‑hot‑encoded features.

y is the continuous numeric target to be predicted by linear regression

* Perform train‑test split (80–20).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


This splits data into 80% training (used to fit the model) and 20% testing (used to evaluate).
​
​

Using random_state ensures reproducible results.

* **Apply feature scaling (if required)**

For linear regression, you usually scale the numerical features only

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numeric columns (those not created by one‑hot)
numeric_cols = ["age", "bmi", "children"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])



This scales age, bmi, and children to have mean ≈ 0 and std ≈ 1.
​
​

The one‑hot encoded columns remain as 0/1 (no scaling).

* Why scaling is / isn’t required.

For this model (Ordinary Least Squares Linear Regression without regularization):
Scaling is not strictly required because OLS is invariant to the scale of features; it only affects the magnitude of coefficients, not predictions.

Where scaling is useful:
If you later use regularized regression (Ridge / Lasso), then scaling becomes important because the penalty term is sensitive to feature scale.
​
​

Scaling also helps with convergence speed in gradient‑based optimizers and makes interpretation of coefficients easier when features are on different scales (e.g., age vs bmi)

SECTION 3 – Model Building.

* Train a Linear Regression model.
* Display:

○ Intercept

○ Coefficients
* Write the regression equation.

*  **Train a Linear Regression mode.**

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize and fit the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)


This fits an ordinary least squares linear regression using all your encoded features to predict charges.

After this, lr_model contains the learned intercept and coefficients.

* Display intercept and coefficients.

In [ ]:
# Intercept
print("Intercept:", lr_model.intercept_)

In [ ]:
# Coefficients as a DataFrame for clarity
import pandas as pd

coeff_df = pd.DataFrame(
    lr_model.coef_,
    index=X_train.columns,
    columns=["Coefficient"]
)
print("\nCoefficients:")
print(coeff_df)

Example output style (numbers will vary with your data):

Intercept: around 1000–3000 (raw‑scale) or higher if not transformed.

Coefficients: each row shows how much charges changes per unit increase in that feature (holding others fixed).



* Write the regression equation.

The general form of the multivariate linear regression equation is

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=coeff_df.index, y='Coefficient', data=coeff_df, palette='viridis', hue=coeff_df.index, legend=False)
plt.title('Linear Regression Coefficients')
plt.xlabel('Features')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = lr_model.predict(X_test)

# Calculate performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"R-squared: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

# SECTION 4 – Model Evaluation

 * Predict on test data.

* Calculate:

○ MAE

○ MSE

○ RMSE

○ R2 Score

* Plot:

○ Actual vs Predicted

○ Residual plot

* Interpret R2 score.

* Predict on test data

After you’ve already trained lr_model on X_train and y_train:

In [ ]:
# Make predictions on test set
y_pred = lr_model.predict(X_test)


y_pred will be the predicted charges for the 20% held‑out test data.

* Calculate MAE, MSE, RMSE, R²

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)


* MAE: average absolute difference between actual and predicted charges; easy to interpret in original units.

* MSE: average of squared errors; penalizes large errors more heavily.

* RMSE: square root of MSE; in same units as charges and often interpreted as “average prediction error”.

* R²: proportion of variance in charges explained by the model (0 = no explanatory power, 1 = perfect fit).

* Typical values for this medical‑cost dataset often show an R² around 0.7–0.8 and an RMSE in the thousands of dollars, depending on scaling and preprocessing.

* Plot: Actual vs Predicted and Residual plot.

Actual vs Predicted

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted Charges")
plt.show()


Points close to the diagonal line indicate good predictions; large deviations show poor fits.



Residual plot

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals (Actual − Predicted)")
plt.title("Residual Plot")
plt.show()


If residuals scatter randomly around zero with no clear pattern, the linearity assumption and constant variance are reasonable. Patterns (e.g., funnel‑shape) suggest issues such as model misspecification or heteroscedasticity.



# Interpret R² score

* R² = 0.75 means your model explains about 75% of the variability in medical charges using the given features (age, BMI, children, smoker status, etc.). The remaining 25% is unexplained noise or factors not included in the model.

* A high R² (close to 1) suggests the model fits the training data well, but you should also check test‑set performance and residual plots to ensure it is not overfitting

#SECTION 5 – Business Insights

* Which variable impacts charges the most?

* How much more do smokers pay?

* Is BMI statistically impactful?

* Can this model be used in production? Why or why not?

**1. Which variable impacts charges the most?**

-In most versions of this dataset, smoker_yes (or the “smoker” variable) has the largest absolute coefficient and explains the biggest jump in charges.

-Even if another feature (like age or bmi) has a large coefficient, the practical impact of being a smoker is usually much bigger because it can add thousands of dollars to predicted charges.

**2. How much more do smokers pay?**

* Use the coefficient for smoker_yes from your model:

-If the coefficient is, say, ≈ 22,000, then the model estimates that smokers pay about ₹22,000 (or equivalent in your currency) more in average charges than non‑smokers, with all other features (age, BMI, children, etc.) held fixed.

**3. Is BMI statistically impactful?**

* To answer this, you need to inspect:

-The coefficient of bmi (how much charges change per unit increase in BMI).

-The p‑value or confidence interval (if your tool / package shows it).

* Typical interpretation:

-If the bmi coefficient is positive and its p‑value is less than 0.05, then BMI is statistically significant: higher BMI is associated with higher charges, even after controlling for other variables.

-If the p‑value is high (e.g., > 0.05), then BMI is not statistically significant in this sample, meaning the data do not provide strong evidence that BMI affects charges once age, smoking, etc., are in the model.

**4. Can this model be used in production? Why or why not?**

* Typical short‑answer structure:

-Yes, with caution, but not out of the box.

* Reasons why it can be useful:

-It gives a simple, interpretable way to estimate medical charges using age, BMI, children, and smoking status, which matches real‑world insurance risk‑factors.

-Metrics like R² ≈ 0.7–0.8 and reasonable RMSE mean the model explains a substantial portion of charge variation for many policyholders.

-Reasons why it should not be used naively in production:

-It may miss important real‑world factors (pre‑existing diseases, surgery type, hospital, location‑specific pricing), which can bias predictions.

-Residuals may show patterns (e.g., high charges for older smokers), so the model should be updated with more data, fairness checks, and regularization before deployment.

-Legal and ethical rules (e.g., “discrimination by age/smoking”) may require constraints or audits before using predictions for pricing or coverage decisions.


#PART B – CLASSIFICATION

**Movie Rating Prediction**

**Objective:** Build a classification model to predict whether a movie is a
Hit or Flop.

**Define:**

● Hit = rating >= 7

● Flop = rating < 7

**SECTION 1 – Data Understanding**

* Load dataset and display:

○ Shape

○ Columns

○ Data types

* Check missing values.

* Create binary target column: Hit

* Show class distribution.

* Plot rating distribution.

○ Write 3 observations.

In [ ]:
import pandas as pd

In [ ]:
# Load dataset (adjust filename)
df = pd.read_csv("/content/sample_data/tmdb_5000_movies.csv")

In [ ]:
# shape
print("Shape:", df.shape)

In [ ]:
# column names
print("Columns:", df.columns.tolist())

In [ ]:
#  data types
print("Data types:")
print(df.dtypes)

-df.shape returns a tuple (rows, columns) showing how many observations and features you have.

-df.columns lists all column names; df.dtypes shows the data type of each column.

* Check missing values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
print("Any missing values?", df.isnull().values.any())


* Create binary target column: Hit
Assuming your dataset has a rating column (say from 1–10 or 1–5), treat high ratings as “Hit”:

In [ ]:
# Create binary target column: Hit (rating >= 7 → Hit = 1; otherwise Hit = 0)
df["Hit"] = (df["vote_average"] >= 7).astype(int)

Adjust the threshold as needed:

-For 1–5 scale: maybe df["rating"] >= 4

-For 1–10 scale: maybe df["rating"] >= 7

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['vote_average'], bins=20, kde=True)
plt.title('Distribution of Movie Vote Averages')
plt.xlabel('Vote Average')
plt.ylabel('Number of Movies')
plt.show()

* Show class distribution

In [ ]:
print("Class distribution (Hit):")
print(df["Hit"].value_counts())
print("Class distribution (percentage):")
print(df["Hit"].value_counts(normalize=True) * 100)


-How many records are Hit = 0 (no‑hit) and Hit = 1 (hit).

-Whether the dataset is balanced (roughly 50–50) or imbalanced (e.g., 90% non‑hit, 10% hit).

* Plot rating distribution; 3 observations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["vote_average"], kde=True)
plt.title("Distribution of Rating")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.show()

1. Rating distribution is skewed: Most items have low or medium ratings, while only a small fraction reach very high ratings, suggesting that “hits” are relatively rare.

2. Clear separation between hits and non‑hits: The chosen threshold (e.g., ≥ 6) cleanly separates a small group of high‑rated items that can be labeled as “Hit”, while the majority fall below it.

3. Limited range / ceiling effect: Ratings are often cluttered near the top of the scale (e.g., many 5s or 10s), which may compress variation and make it harder to distinguish between “very good” and “okay” items just from rating alone.



#SECTION 2 – Data Preprocessing

* Drop irrelevant columns.

* Encode categorical variables.

* Separate X and y.

* Train-test split (80-20).

* Scale if required.

* Drop irrelevant columns

-Keep only features that are likely to affect the outcome and drop metadata or identifiers.

In [ ]:
#  drop ID, name, date, or any obviously irrelevant columns
irrelevant_cols = ["id", "name", "timestamp"]   # adapt to your dataset
df = df.drop(columns=irrelevant_cols, errors="ignore")


* A good rule:

-Keep numeric features (e.g., rating, views, duration, budget) and meaningful categoricals (e.g., genre, country).

-Drop redundant or purely tracking columns that won’t help predict “Hit”.

*  **Encode categorical variables**

Convert categorical columns (like genre, country, etc.) into numeric form using one‑hot encoding

In [ ]:
# Identify categorical columns (object / category types)
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Exclude the target 'Hit' if it somehow appears here
cat_cols = [col for col in cat_cols if col != "Hit"]

# One‑hot encode
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)



-drop_first=True avoids perfect multicollinearity (one column per category is enough).

-The result will be several new binary columns (e.g., genre_comedy, genre_drama, country_usa, etc.).

* **Separate X and y**

In [ ]:
X = df.drop("Hit", axis=1)   # all feature columns
y = df["Hit"]               # binary target (0 or 1)


-X contains all predictors (numerics + one‑hot dummies).

-y is the binary label we want to predict.

* **Train‑test split (80–20)**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preserves class balance in train/test
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


-stratify=y keeps the same proportion of Hit = 0 and Hit = 1 in both train and test, which is important for evaluation. [web]

* **Scale if required**

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify only numeric columns (not dummies)
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])


-Use X_train and X_test raw for tree models;

-Use X_train_scaled and X_test_scaled for models sensitive to feature scale

#SECTION 3 – Model Building
* Train Logistic Regression model.

* Display model coefficients.

* Explain what coefficients mean.

* Train Logistic Regression model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer # Import SimpleImputer

# Use X_train and y_train from Section 2

# Identify numerical columns for imputation
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

# Create an imputer and fit on X_train, then transform both X_train and X_test
imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = imputer.transform(X_test[numeric_cols])

lr_model = LogisticRegression(random_state=42, max_iter=1000) # Increased max_iter for convergence
lr_model.fit(X_train, y_train)

*  **Display model coefficients**

In [ ]:
# Intercept
print("Intercept:", lr_model.intercept_[0])

In [ ]:
# Coefficients
import pandas as pd

coeff_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_model.coef_[0]
})

print("\nCoefficients:")
print(coeff_df.sort_values(by="Coefficient", key=abs, ascending=False))

-The intercept is the log‑odds of “Hit” when all features are 0.

-Each coefficient shows how the log‑odds of Hit = 1 changes when that feature increases by 1 unit (or when that category is present, in the case of one‑hot encoded variables).

* **Explain what coefficients mean**

* In logistic regression:

-A positive coefficient means that increasing that feature (or having that category) increases the log‑odds of Hit = 1, i.e., makes the item more likely to be a hit.

-A negative coefficient means that increasing that feature decreases the log‑odds of Hit = 1, i.e., makes an item less likely to be a hit.

-The larger the absolute value of the coefficient, the stronger the effect of that feature on the outcome.

#SECTION 4 – Evaluation
* Predict test data.

* Calculate:

○ Accuracy

○ Precision

○ Recall

○ F1 Score

* Plot Confusion Matrix.

* Explain:

○ Why accuracy alone is not enough?

○ Which metric matters more here?

* Predict test data

In [ ]:
# Predict class labels (0 or 1)
y_pred = lr_model.predict(X_test)

In [ ]:
# Optionally also get probabilities
y_pred_proba = lr_model.predict_proba(X_test)[:, 1]

-y_pred contains the model’s binary predictions for the 20% test set.

-y_pred_proba is useful if you later want to adjust the classification threshold

* **Calculate Accuracy, Precision, Recall, F1**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)


-Accuracy: overall fraction of correct predictions (both 0 and 1).

-Precision: fraction of predicted “Hits” that are actually Hits (TP / (TP + FP)).

-Recall: fraction of actual Hits that the model caught (TP / (TP + FN)).

-F1: harmonic mean of precision and recall; balances the two.

*  **Plot Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Hit (0)", "Hit (1)"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


* The matrix shows:

-Top‑left: True Negatives (correctly predicted No Hit).

-Top‑right: False Positives (predicted Hit but actually No Hit).

-Bottom‑left: False Negatives (missed Hits).

-Bottom‑right: True Positives (correctly predicted Hits).



* Which metric matters more here?

-For a “Hit” (success / popularity) prediction task, recall is usually more important, because:

-The business wants to identify as many true Hits as possible (e.g., to promote winners, allocate budget, or sign contracts).

-A higher recall means fewer missed Hits, even if some non‑Hits are also flagged (precision may drop slightly).

* Why accuracy alone is not enough?

-Accuracy can be misleading when classes are imbalanced.

-For example, if 90% of items are No Hit and only 10% are Hit, a model that always predicts No Hit will have ~90% accuracy but completely fails to detect any real Hits.

-Accuracy doesn’t distinguish between false positives and false negatives, which can have very different business costs.

-In a “Hit” prediction system, missing a real Hit (false negative) may mean missing a profitable opportunity, while a false positive usually only means some extra review or investment.

#SECTION 5 – Business Interpretation
* What factors influence movie success?
* Is the dataset balanced?
* Would you trust this model for production? Why?

* What factors influence movie success?

-In your model, the most influential factors are the features with largest absolute coefficients (especially positive ones). Common examples:

-High rating → strongly increases the probability of being a “Hit”.

-Specific genres (e.g., action, comedy, drama) → some genres are associated with higher success.

-Moderate runtime / budget → very long movies or extremely expensive ones may reduce chances if those coefficients are negative.

-Belonging to a successful region (e.g., large‑market countries) → boosts the odds of being a Hit.

* **Is the dataset balanced?**

-Use the class‑distribution calculation from Section 1:

In [ ]:
print(df["Hit"].value_counts())
print(df["Hit"].value_counts(normalize=True) * 100)


-If the two classes are roughly 50–50 (e.g., 45–55%), the dataset is balanced.

-If one class is much smaller (e.g., 80–90% No Hit, 10–20% Hit), the dataset is imbalanced.

* **Would you trust this model for production? Why?**

-Short answer: conditionally yes, but not in its current raw form.

* **Why it can be trusted (with caveats):**

-It uses interpretable logistic regression, so you can see which features increase or decrease the chance of a Hit and explain decisions to stakeholders.

-If accuracy, recall, and F1 are reasonably high (e.g., F1 > 0.7) and the confusion matrix is acceptable, the model captures a meaningful pattern in the data.

* **Why it might not be ready for production yet:**

-If the dataset is imbalanced, the model may still miss many true Hits in the real world, unless you tune the probability threshold or use class‑weighted training.

-It may not include important business factors (e.g., release date, marketing budget, seasonality, competition), so predictions can be incomplete.

* **Before production use, you should:**

-Validate on fresh data and monitor over time.

-Check for bias (e.g., certain genres or regions always labeled “No Hit”) and adjust for fairness.